In [16]:
# pip install transformers==4.44.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 15.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 19.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
Note: you may need to restart the kernel to use updated packages.


In [15]:
# pip uninstall transformers -y

Found existing installation: transformers 4.57.0
Uninstalling transformers-4.57.0:
  Successfully uninstalled transformers-4.57.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
)

path = 'OpenGVLab/InternVL3-9B'
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    low_cpu_mem_usage=True,
    use_flash_attn=False,
    trust_remote_code=True,
    device_map="auto").eval()
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)

FlashAttention2 is not installed.


Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████| 4/4 [00:05<00:00,  1.47s/it]


In [6]:
import pandas as pd
df = pd.read_csv('../../LAGENDA/lagenda_annotation.csv')

imgs_todas_idades_ok = (
    df.groupby('img_name')['age']
      .apply(lambda x: (x != -1).all())
      .reset_index()
)

# Filtra as imagens em que TODAS as idades são conhecidas
imgs_todas_idades_ok = imgs_todas_idades_ok[imgs_todas_idades_ok['age'] == True]

# tem uma imagem faltando nos arquivos (olhar depois)
imgs_todas_idades_ok = imgs_todas_idades_ok[imgs_todas_idades_ok['img_name'] != 'lag_benchmark/b5c8ef090f134ad5.jpg.jpg']

print(f"{len(imgs_todas_idades_ok)} imagens têm todas as idades conhecidas.")
display(imgs_todas_idades_ok)

20182 imagens têm todas as idades conhecidas.


,img_name,age
0,lag_benchmark/0000339d0372e7e6.jpg,True
6,lag_benchmark/0004350a376865f5.jpg,True
9,lag_benchmark/00059fa95ee95e65.jpg,True
11,lag_benchmark/0007473cdbe8c18d.jpg,True
12,lag_benchmark/000851174d48d3d9.jpg,True
...,...,...
67154,lag_benchmark/fff74322fa07c7c1.jpg,True
67155,lag_benchmark/fff934bd1ff95af5.jpg,True
67156,lag_benchmark/fffb40d02f510b35.jpg,True
67157,lag_benchmark/ffff1ad4bf685255.jpg,True


In [3]:
prompt = """USER: <image>
Você recebe uma imagem que pode conter uma ou mais pessoas. 
Para CADA pessoa determine a sua idade exata a partir de sua aparência.

Regras:
- Responda APENAS com JSON válido
- Nenhum texto extra antes ou depois
- A idade deve ser um número inteiro.
- Estime individualmente a idade de CADA pessoa.

Formato de saída:
{
  "número de pessoas": <inteiro>,
  "pessoas": [
    {"id": 1, "idade": <idade da pessoa> }, 
  ]
}
ASSISTANT:"""

In [4]:
import os
import json
import requests
from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tqdm import tqdm  # <-- barra de progresso
from codecarbon import EmissionsTracker
from torchvision.transforms.functional import InterpolationMode

import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD

    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])

    return transform
    
def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values




In [7]:
########## code carbon ##########
tracker = EmissionsTracker(tracking_mode='process', log_level='critical', output_file='emissions_internVL3.csv')
tracker.start()
#################################

# pasta_imagens = "../../LAGENDA/images"
pasta_imagens = "../../LAGENDA/images"

resultados = []
batch_size = 8
cont = 0

torch.cuda.empty_cache()
for i in tqdm(range(0, len(imgs_todas_idades_ok), batch_size), desc="Processando imagens"):
    batch_imgs = imgs_todas_idades_ok['img_name'][i:i+batch_size]
    
    batch_pixel_values = []
    num_patches_list = []
    batch_nomes_arquivo = []
    
    for img_name in batch_imgs:
        nome_arquivo = img_name.split("/")[-1]
        batch_nomes_arquivo.append(nome_arquivo)
        caminho = f"{pasta_imagens}/{nome_arquivo}"
        pixel_values = load_image(caminho, max_num=1) #processa a imagem, dividindo em patches(pedaços da imagem)
        num_patches_list.append(pixel_values.size(0)) #lista de quantos patches cada imagem possui.
        pixel_values = pixel_values.to(torch.bfloat16).cuda()
        batch_pixel_values.append(pixel_values)
    pixel_values = torch.cat(batch_pixel_values, dim=0) 

    responses = model.batch_chat(tokenizer, pixel_values,
                             num_patches_list=num_patches_list,
                             questions= [prompt] * len(batch_imgs),
                             generation_config = {"max_new_tokens": 200, "do_sample": False})
    for img_name, response in zip(batch_nomes_arquivo, responses):
        resultados.append({
            "img_name": img_name,
            "resposta_modelo": response
        })

        
    del pixel_values
    torch.cuda.empty_cache()

########## code carbon ##########
emissions = tracker.stop()
print(f"Emissions: {emissions} kg CO₂")
#################################
    
# visualizar
df_resultados = pd.DataFrame(resultados)
display(df_resultados)

df_resultados.to_csv('resultados_internvl3_json.csv', index=False)


Processando imagens: 100%|███████████████████████████████████████████████████████| 2523/2523 [21:41:29<00:00, 30.95s/it]

Emissions: 0.6539044284217981 kg CO₂


,img_name,resposta_modelo
0,0000339d0372e7e6.jpg,"{\n ""número de pessoas"": 1,\n ""pessoas"": [\n..."
1,0004350a376865f5.jpg,"{\n ""número de pessoas"": 2,\n ""pessoas"": [\n..."
2,00059fa95ee95e65.jpg,"{\n ""número de pessoas"": 1,\n ""pessoas"": [\n..."
3,0007473cdbe8c18d.jpg,"{\n ""número de pessoas"": 1,\n ""pessoas"": [\n..."
4,000851174d48d3d9.jpg,"{\n ""número de pessoas"": 1,\n ""pessoas"": [\n..."
...,...,...
20177,fff74322fa07c7c1.jpg,"{\n ""número de pessoas"": 1,\n ""pessoas"": [\n..."
20178,fff934bd1ff95af5.jpg,"{\n ""número de pessoas"": 1,\n ""pessoas"": [\n..."
20179,fffb40d02f510b35.jpg,"{\n ""número de pessoas"": 1,\n ""pessoas"": [\n..."
20180,ffff1ad4bf685255.jpg,"{\n ""número de pessoas"": 1,\n ""pessoas"": [\n..."


In [7]:
resultados

[{'img_name': '0000339d0372e7e6.jpg',
  'resposta_modelo': '{\n  "número de pessoas": 1,\n  "pessoas": [\n    {"id": 1, "idade": 30}\n  ]\n}'},
 {'img_name': '0004350a376865f5.jpg',
  'resposta_modelo': '{\n  "número de pessoas": 2,\n  "pessoas": [\n    {"id": 1, "idade": 5},\n    {"id": 2, "idade": 4}\n  ]\n}'},
 {'img_name': '00059fa95ee95e65.jpg',
  'resposta_modelo': '{\n  "número de pessoas": 1,\n  "pessoas": [\n    {"id": 1, "idade": 60}\n  ]\n}'},
 {'img_name': '0007473cdbe8c18d.jpg',
  'resposta_modelo': '{\n  "número de pessoas": 1,\n  "pessoas": [\n    {"id": 1, "idade": 5}\n  ]\n}'},
 {'img_name': '000851174d48d3d9.jpg',
  'resposta_modelo': '{\n  "número de pessoas": 1,\n  "pessoas": [\n    {"id": 1, "idade": 3}\n  ]\n}'},
 {'img_name': '000de3ca54bd4c1a.jpg',
  'resposta_modelo': '{\n  "número de pessoas": 1,\n  "pessoas": [\n    {"id": 1, "idade": 30}\n  ]\n}'},
 {'img_name': '00107eaf96d165ca.jpg',
  'resposta_modelo': '{\n  "número de pessoas": 1,\n  "pessoas": [\n    {

In [4]:
# prompts_mais_jovem = [
#     """<image> Há alguma criança (menor de 14 anos) presente na imagem? Se sim, responda com 1.Caso contrário, responda com 0.
#     ASSISTANT:""",
#     """<image> Há alguma criança (menor de 18 anos) presente na imagem? Se sim, responda com 1. Caso contrário, responda com 0.
#     ASSISTANT:""",
#     """<image> Identifique a pessoa mais jovem na imagem e classifique-a em uma das seguintes faixas etárias: criança (0–13), adolescente (14–17), adulto (18–59) ou idoso (60–96).Responda somente com o nome da faixa etária dessa pessoa.
#     ASSISTANT:""",
#     """<image> Qual a idade, em anos, da pessoa mais jovem presente na imagem?.Responda com um único número inteiro que represente essa idade.
#     ASSISTANT:"""
# ]